# UD03 Tarea con gemini

## Enunciado

Ahora que ya conocemos todo el ecosistema Hadoop-Spark en profundidad, vamos a hacer un ejercicio que abarque todo el proceso de tratamiento de los datos: desde que se extraen de un repositorio, se limpian, se trasforman, se exportan a una BD (en este caso relacional) y se realizan consultas sobre ellos.

Para ello una vez descargado el data set:


Debes entregar un cuaderno de Colab con:

Explicación del conjunto de datos elegido cuál es su temática, su tamaño, con qué campos se relaciona y por qué puede resultar interesante.  
Una vez quede claro el contexto:

1. Importa de manera automática tus conjuntos de datos. Realizado con archivo en drive.  
2. Usa Pandas para realizar una primera visualización de ellos en forma de tabla, y estudiar los datos: observar si hay valores nulos, fechas con formatos que no se corresponden, campos vacíos...   
3. Usa Apache Pig para  
   1. corregir los fallos en los datos que hayas observado en el punto anterior. (Al menos debes modificar dos campos, por ejemplo cambiar formatos de fechas y rellenar campos vacíos y nulos con valores por defecto)  
   2. Realizar un tratamiento de datos que consideres interesante: por ejemplo, encontrar las 3 palabras más repetidas en ambos archivos.  
4. Usa Spark (PySpark) para importar tus ficheros a una base de datos relacional Hive.  
5. Realiza al menos dos consultas HQL que impliquen dos tablas.

# Instalar Hadoop y Pig

Lo primero que hago es activar el soporte de GPU.

**"Entorno de ejecución" > "Cambiar tipo de entorno de ejecución"**

## Verificaciones previas

Primero, verifico la versión de Java:

In [1]:
!ls -l /usr/lib/jvm/

total 4
lrwxrwxrwx 1 root root   21 Jan 29 03:35 java-1.17.0-openjdk-amd64 -> java-17-openjdk-amd64
drwxr-xr-x 9 root root 4096 Apr  2 13:13 java-17-openjdk-amd64


## Instalación de Hadoop

Entro a https://downloads.apache.org/hadoop/common y veo las versiones disponibles para descargar.

Una vez comprobado qué versión de Java necesito para la versión de Hadoop que usaré, descargo el tar.gz correspondiente.

Para saberlo, consulto el archivo Changelog y busco "Java". Ahí veo frases como:

"[JDK17] Add JPMS options required by Java 17+"

In [2]:
import os

# Download and install Hadoop 3.4.2
!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz
!tar -xzf hadoop-3.4.2.tar.gz
# Remove destination if it exists to avoid errors or nesting
!rm -rf /usr/local/hadoop-3.4.2
!mv hadoop-3.4.2 /usr/local/
!rm hadoop-3.4.2.tar.gz

# Update environment variables for Java 17 and Hadoop 3.4.2
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/usr/local/hadoop-3.4.2"

# Update PATH to include Hadoop bin
if "/usr/local/hadoop-3.4.2/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = os.environ["PATH"] + ":" + "/usr/local/hadoop-3.4.2/bin"

# Verify installations
print("Hadoop Version:")
!hadoop version
print("\nJava Version:")
!java -version

Hadoop Version:
Hadoop 3.4.2
Source code repository https://github.com/apache/hadoop.git -r 84e8b89ee2ebe6923691205b9e171badde7a495c
Compiled by ahmarsu on 2025-08-20T10:30Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum fa94c67d4b4be021b9e9515c9b0f7b6
This command was run using /usr/local/hadoop-3.4.2/share/hadoop/common/hadoop-common-3.4.2.jar

Java Version:
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [3]:
ls -l $HADOOP_HOME/etc/hadoop

total 184
-rw-r--r-- 1 1001 1001  9213 Aug 20  2025 capacity-scheduler.xml
-rw-r--r-- 1 1001 1001  1335 Aug 20  2025 configuration.xsl
-rw-r--r-- 1 1001 1001  2567 Aug 20  2025 container-executor.cfg
-rw-r--r-- 1 1001 1001   774 Aug 20  2025 core-site.xml
-rw-r--r-- 1 1001 1001  3999 Aug 20  2025 hadoop-env.cmd
-rw-r--r-- 1 1001 1001 16786 Aug 20  2025 hadoop-env.sh
-rw-r--r-- 1 1001 1001  3321 Aug 20  2025 hadoop-metrics2.properties
-rw-r--r-- 1 1001 1001 14007 Aug 20  2025 hadoop-policy.xml
-rw-r--r-- 1 1001 1001  3414 Aug 20  2025 hadoop-user-functions.sh.example
-rw-r--r-- 1 1001 1001   683 Aug 20  2025 hdfs-rbf-site.xml
-rw-r--r-- 1 1001 1001   775 Aug 20  2025 hdfs-site.xml
-rw-r--r-- 1 1001 1001  1484 Aug 20  2025 httpfs-env.sh
-rw-r--r-- 1 1001 1001  1657 Aug 20  2025 httpfs-log4j.properties
-rw-r--r-- 1 1001 1001   620 Aug 20  2025 httpfs-site.xml
-rw-r--r-- 1 1001 1001  3518 Aug 20  2025 kms-acls.xml
-rw-r--r-- 1 1001 1001  1351 Aug 20  2025 kms-env.sh
-rw-r--r-- 1 1001 1001 

## Instalación de Pig y configuración de entrono

In [4]:
%%bash
wget https://downloads.apache.org/pig/pig-0.17.0/pig-0.17.0.tar.gz
tar -xzf pig-0.17.0.tar.gz
cp -r pig-0.17.0/ /usr/local/

--2026-04-22 19:57:31--  https://downloads.apache.org/pig/pig-0.17.0/pig-0.17.0.tar.gz
Resolving downloads.apache.org (downloads.apache.org)... 135.181.214.104, 88.99.208.237, 2a01:4f8:10a:39da::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|135.181.214.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 230606579 (220M) [application/x-gzip]
Saving to: ‘pig-0.17.0.tar.gz’

     0K .......... .......... .......... .......... ..........  0%  159K 23m40s
    50K .......... .......... .......... .......... ..........  0%  317K 17m44s
   100K .......... .......... .......... .......... ..........  0% 18.5M 11m53s
   150K .......... .......... .......... .......... ..........  0% 17.1M 8m58s
   200K .......... .......... .......... .......... ..........  0%  325K 9m29s
   250K .......... .......... .......... .......... ..........  0% 19.3M 7m56s
   300K .......... .......... .......... .......... ..........  0% 20.0M 6m49s
   350K .......... .....

Establecemos las variables de entorno para Pig

In [5]:
import os
# Update PIG_CLASSPATH
os.environ["PIG_HOME"] = "/usr/local/pig-0.17.0"
os.environ["PATH"] = os.environ["PATH"] + ":" + "/usr/local/pig-0.17.0/bin"
os.environ["PIG_CLASSPATH"] = "/usr/local/hadoop-3.4.2/etc/hadoop"

Verificamos la instalación

In [6]:
!pig -h -version


Apache Pig version 0.17.0 (r1797386) 
compiled Jun 02 2017, 15:41:58

USAGE: Pig [options] [-] : Run interactively in grunt shell.
       Pig [options] -e[xecute] cmd [cmd ...] : Run cmd(s).
       Pig [options] [-f[ile]] file : Run cmds found in file.
  options include:
    -4, -log4jconf - Log4j configuration file, overrides log conf
    -b, -brief - Brief logging (no timestamps)
    -c, -check - Syntax check
    -d, -debug - Debug level, INFO is default
    -e, -execute - Commands to execute (within quotes)
    -f, -file - Path to the script to execute
    -g, -embedded - ScriptEngine classname or keyword for the ScriptEngine
    -h, -help - Display this message. You can specify topic to get help for that topic.
        properties is the only topic currently supported: -h properties.
    -i, -version - Display version information
    -l, -logfile - Path to client side log file; default is current working directory.
    -m, -param_file - Path to the parameter file
    -p, -param - K

# 1.- **Apartado de Importación de datos**
---
Importa de manera automática tus conjuntos de datos desde Kaggle u otro repositorio (Pista: puedes usar las librerías kaggle, opendatests u otras opciones que consideres más oportunas, también puedes usar squoop, dependiendo de dónde se encuentren tus datos).
---
---

## Clonar repo con dataset (alternative para examen)

In [7]:
!git clone https://github.com/kachytronico/BDA_examen_26

Cloning into 'BDA_examen_26'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 92 (delta 28), reused 77 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 1.41 MiB | 12.67 MiB/s, done.
Resolving deltas: 100% (28/28), done.


## Descarga desde archivo de Drive

Con el bloque de código que aparece a continuación, descargamos los ficheros que hemos alojado en nuestro Google Drive y los ubicamos en la ruta de Colab que vamos a emplear en los próximos pasos.

Además de los tres archivos CSV mencionados en el punto anterior, como el fichero *transactions_data.csv* tiene **más de 13 millones de líneas** y puede dar problemas de memoria RAM o tardar mucho en la ejecución, hemos subido también una versión CORTA.

In [8]:
# Descargamos un fichero comprimido con cuatro archivos desde Google Drive usando el ID del archivo
!gdown --id 1_6ptGvYG_tgllD3axsE6FQT4ZYWxgKJ9 -O datasets.zip

# Descomprimimos el archivo ZIP descargado en la carpeta "datasets"
!unzip datasets.zip -d datasets/

import os

# Eliminamos el archivo ZIP descargado
if os.path.exists("datasets.zip"):
    os.remove("datasets.zip")
    print("El archivo datasets.zip ha sido eliminado.")
else:
    print("El archivo datasets.zip no existe.")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1_6ptGvYG_tgllD3axsE6FQT4ZYWxgKJ9
From (redirected): https://drive.google.com/uc?id=1_6ptGvYG_tgllD3axsE6FQT4ZYWxgKJ9&confirm=t&uuid=7b719c30-4e6b-4737-a497-9b77f68b4907
To: /content/datasets.zip
100% 312M/312M [00:01<00:00, 160MB/s]
Archive:  datasets.zip
  inflating: datasets/users_data.csv  
  inflating: datasets/cards_data.csv  
  inflating: datasets/transactions_data.csv  
  inflating: datasets/transactionsCORTO_data.csv  
El archivo datasets.zip ha sido eliminado.


### Comprobación de archivos importados

Verificamos que los archivos CSV se encuentran en la carpeta `datasets`.

In [20]:
import os

# Listar el contenido del directorio 'datasets'
files_in_datasets = os.listdir('datasets')
print("Archivos en la carpeta 'datasets':")
for file_name in files_in_datasets:
    print(file_name)

Archivos en la carpeta 'datasets':
cards_data.csv
transactions_data.csv
users_data.csv
transactionsCORTO_data.csv


## Mover Datasets Locales a HDFS

Ahora que Hadoop y Pig están instalados y configurados, el siguiente paso es mover los archivos CSV que hemos descargado localmente a un directorio en HDFS, en este caso, llamado `/input`. Esto es necesario para que Pig pueda procesarlos.

In [21]:
# Crear el directorio /input en HDFS
!hdfs dfs -mkdir -p input

# Copiar los archivos desde la ruta local /content/datasets/ a /input en HDFS
!hdfs dfs -put /content/datasets/* input

# Verificar que los archivos se encuentran correctamente en HDFS
!hdfs dfs -ls input

put: `input/cards_data.csv': File exists
put: `input/transactionsCORTO_data.csv': File exists
put: `input/transactions_data.csv': File exists
put: `input/users_data.csv': File exists
Found 4 items
-rw-r--r--   1 root root     509860 2026-04-22 19:58 input/cards_data.csv
-rw-r--r--   1 root root     951089 2026-04-22 19:58 input/transactionsCORTO_data.csv
-rw-r--r--   1 root root 1258531040 2026-04-22 19:59 input/transactions_data.csv
-rw-r--r--   1 root root     164831 2026-04-22 19:59 input/users_data.csv


### Visualización de las primeras líneas de los archivos en HDFS

In [22]:
# Mostrar las primeras líneas de cards_data.csv
!hdfs dfs -cat input/cards_data.csv | head -n 5

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
cat: Unable to write to output stream.


In [23]:
# Mostrar las primeras líneas de transactionsCORTO_data.csv
!hdfs dfs -cat input/transactionsCORTO_data.csv | head -n 5

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,
7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,
7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,
7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,
cat: Unable to write to output stream.


In [24]:
# Mostrar las primeras líneas de transactions_data.csv
!hdfs dfs -cat input/transactions_data.csv | head -n 5

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,
7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,
7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,
7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,
cat: Unable to write to output stream.


In [25]:
# Mostrar las primeras líneas de users_data.csv
!hdfs dfs -cat input/users_data.csv | head -n 5

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
cat: Unable to write to output stream.


### Conclusión del Punto 1: Importación de Datos

El primer punto del enunciado, referente a la importación automática de los conjuntos de datos, se ha completado satisfactoriamente.

Se han realizado los siguientes pasos:

1.  **Descarga y Extracción:** Los archivos de datos (`users_data.csv`, `cards_data.csv`, `transactions_data.csv`, y `transactionsCORTO_data.csv`) han sido descargados de Google Drive y descomprimidos en el entorno local de Colab dentro de la carpeta `/content/datasets/`.
2.  **Verificación Local:** Se ha confirmado la existencia de los archivos en la ruta local.
3.  **Transferencia a HDFS:** Todos los archivos CSV han sido copiados exitosamente desde la ubicación local (`/content/datasets/`) al sistema de archivos distribuido de Hadoop (HDFS) en el directorio `/input`.
4.  **Verificación en HDFS:** Se ha verificado que los archivos están correctamente listados en HDFS y se han mostrado las primeras líneas de cada uno para una inspección inicial del formato y contenido.

Con esto, los datos están ahora disponibles en HDFS para ser procesados por herramientas como Pig y Spark, cumpliendo con el requisito de importación del ejercicio.

# 2.- Primera Visualización y Estudio de Datos con Pandas

En este apartado, utilizaremos la librería Pandas para cargar los datasets en DataFrames, visualizarlos y realizar una primera inspección para identificar posibles problemas como valores nulos, formatos incorrectos o campos vacíos.

In [26]:
import pandas as pd

# Función para leer archivos de HDFS a un DataFrame local
def read_hdfs_csv_to_pandas(hdfs_path, local_path, separator=','):
    !hdfs dfs -get -f {hdfs_path} {local_path}
    df = pd.read_csv(local_path, sep=separator)
    # Limpiar el archivo local después de leerlo para ahorrar espacio
    !rm {local_path}
    return df

### Inspección de `cards_data.csv`

In [27]:
print('Cargando cards_data.csv...')
cards_df = read_hdfs_csv_to_pandas('input/cards_data.csv', 'cards_data.csv')

print('\nPrimeras 5 filas de cards_data.csv:')
print(cards_df.head())

print('\nInformación del DataFrame cards_data.csv:')
cards_df.info()

print('\nValores nulos en cards_data.csv:')
print(cards_df.isnull().sum())

Cargando cards_data.csv...

Primeras 5 filas de cards_data.csv:
     id  client_id  card_brand        card_type       card_number  expires  \
0  4524        825        Visa            Debit  4344676511950444  12/2022   
1  2731        825        Visa            Debit  4956965974959986  12/2020   
2  3701        825        Visa            Debit  4582313478255491  02/2024   
3    42        825        Visa           Credit  4879494103069057  08/2024   
4  4659        825  Mastercard  Debit (Prepaid)  5722874738736011  03/2009   

   cvv has_chip  num_cards_issued credit_limit acct_open_date  \
0  623      YES                 2       $24295        09/2002   
1  393      YES                 2       $21968        04/2014   
2  719      YES                 2       $46414        07/2003   
3  693       NO                 1       $12400        01/2003   
4   75      YES                 1          $28        09/2008   

   year_pin_last_changed card_on_dark_web  
0                   2008        

### Inspección de `users_data.csv`

In [28]:
print('Cargando users_data.csv...')
users_df = read_hdfs_csv_to_pandas('input/users_data.csv', 'users_data.csv')

print('\nPrimeras 5 filas de users_data.csv:')
print(users_df.head())

print('\nInformación del DataFrame users_data.csv:')
users_df.info()

print('\nValores nulos en users_data.csv:')
print(users_df.isnull().sum())

Cargando users_data.csv...

Primeras 5 filas de users_data.csv:
     id  current_age  retirement_age  birth_year  birth_month  gender  \
0   825           53              66        1966           11  Female   
1  1746           53              68        1966           12  Female   
2  1718           81              67        1938           11  Female   
3   708           63              63        1957            1  Female   
4  1164           43              70        1976            9    Male   

                    address  latitude  longitude per_capita_income  \
0             462 Rose Lane     34.15    -117.76            $29278   
1    3606 Federal Boulevard     40.76     -73.74            $37891   
2           766 Third Drive     34.02    -117.89            $22681   
3          3 Madison Street     40.71     -73.99           $163145   
4  9620 Valley Stream Drive     37.76    -122.44            $53797   

  yearly_income total_debt  credit_score  num_credit_cards  
0        $59696

### Inspección de `transactionsCORTO_data.csv`

Utilizaremos la versión 'CORTO' del dataset de transacciones para la inspección inicial con Pandas, debido al gran tamaño del archivo `transactions_data.csv` completo, que podría causar problemas de memoria en este entorno.

In [29]:
print('Cargando transactionsCORTO_data.csv...')
transactions_corto_df = read_hdfs_csv_to_pandas('input/transactionsCORTO_data.csv', 'transactionsCORTO_data.csv')

print('\nPrimeras 5 filas de transactionsCORTO_data.csv:')
print(transactions_corto_df.head())

print('\nInformación del DataFrame transactionsCORTO_data.csv:')
transactions_corto_df.info()

print('\nValores nulos en transactionsCORTO_data.csv:')
print(transactions_corto_df.isnull().sum())

Cargando transactionsCORTO_data.csv...

Primeras 5 filas de transactionsCORTO_data.csv:
        id                 date  client_id  card_id   amount  \
0  7475327  2010-01-01 00:01:00       1556     2972  $-77.00   
1  7475328  2010-01-01 00:02:00        561     4575   $14.57   
2  7475329  2010-01-01 00:02:00       1129      102   $80.00   
3  7475331  2010-01-01 00:05:00        430     2860  $200.00   
4  7475332  2010-01-01 00:06:00        848     3915   $46.41   

            use_chip  merchant_id merchant_city merchant_state      zip   mcc  \
0  Swipe Transaction        59935        Beulah             ND  58523.0  5499   
1  Swipe Transaction        67570    Bettendorf             IA  52722.0  5311   
2  Swipe Transaction        27092         Vista             CA  92084.0  4829   
3  Swipe Transaction        27092   Crown Point             IN  46307.0  4829   
4  Swipe Transaction        13051       Harwood             MD  20776.0  5813   

  errors  
0    NaN  
1    NaN  
2    Na

### Conclusión del Punto 2: Primera Visualización y Estudio de Datos

Tras la primera visualización y estudio de los datasets con Pandas, se han identificado los siguientes puntos clave:

*   **`cards_data.csv` y `users_data.csv`:** Ambos datasets no presentan valores nulos. Sin embargo, contienen columnas como `expires`, `acct_open_date`, `credit_limit`, `per_capita_income`, `yearly_income` y `total_debt` que son de tipo `object` (cadena de texto) y que requieren ser convertidas a tipos de datos más adecuados (fechas o numéricos) para su correcto análisis. Los campos monetarios (`credit_limit`, `per_capita_income`, `yearly_income`, `total_debt`) necesitan que se elimine el símbolo '$' antes de la conversión.

*   **`transactionsCORTO_data.csv`:** Este dataset muestra varios problemas:
    *   **Valores Nulos:** Las columnas `merchant_state` (1179 nulos), `zip` (1205 nulos) y `errors` (9848 nulos) contienen una cantidad significativa de valores faltantes que deberán ser tratados (rellenados o eliminados).
    *   **Tipos de Datos y Formato:** Las columnas `date` (tipo `object`) y `amount` (tipo `object` debido al símbolo '$' y signos negativos) necesitan ser convertidas a tipos de datos apropiados (fecha/hora y numérico, respectivamente).

En resumen, el siguiente paso será corregir estos problemas de calidad de datos, especialmente la gestión de valores nulos y la conversión de tipos, antes de proceder con análisis más profundos.